In [2]:
import os
import json
# from notebook.utils.calculate_average_score import ckpt_list
import numpy as np
from collections import defaultdict

base_model = "sd-3-5-medium"
rl_framework = "diffusion-dpo"
dataset_name = "drawbench-unique" # "pick_a_pic_v2"
base_average_score_dir = f"/data_center/data2/dataset/chenwy/21164-data/{rl_framework}/{base_model}"
seed_list = [42, 123, 456, 789, 1000]

# method_list = [ 
#             #    "sd-v1-5",
#             #    "irl_top_512_images-no_anime-hpdv3_all-uids-ckpt_1600-dpo_2000",
#             #    "irl_top_512_images-no_anime-pickscore_0.02-hpdv3_all-uids-ckpt_1600-dpo_2000",
#                "irl_top_512_images_no_anime_colorfulness_pickscore_0.02-hpdv3_all_ckpt_1600-dpo_2000_top_512_images_no_anime_colorfulness_pickscore_0.02-hpdv3_all_inpainting",
#             # "irl_top_512_images-no_anime-hpdv3_all-uids",
#             # "irl_top_512_images-no_anime-pickscore_0.02-hpdv3_all-uids",
#                ]
# ckpt_list = [ 800 ]

method_list = [ 
               "sd-3-5-medium",
               "pick-a-pic-v2-dpo_dataset_160000_pairs",
               "FlowGRPO-PickScore",
               "irl_top_512_images_no_anime_colorfulness_pickscore_0.02-hpdv3_all_lr_0.0002_ckpt_3200-dpo_top_512_images_no_anime_colorfulness_pickscore_0.02-hpdv3_all"
               ]
ckpt_list = [ 0, 4900, 0, 450 ]

# evaluation_metrics = [ "color-fidelity-metric" ]
evaluation_metrics = ["pickscore", "imagereward", "unifiedreward", "hpsv3", "deqa", "aesthetic", "color-fidelity-metric" ]
# evaluation_metrics = ["GLCM_contrast",  "laplacian_variance", "edge_density", "SGP-PickScore", "SGP-HPSv3" ]
# 存储每个方法的所有 seed 的分数
all_method_scores = {}

# 遍历每个方法
for method, ckpt in zip(method_list, ckpt_list):
    print(f"\n{'='*80}")
    print(f"Processing method: {method} (ckpt: {ckpt})")
    print(f"{'='*80}")
    
    # 存储当前方法所有 seed 的分数
    method_scores = defaultdict(list)
    
    # 遍历所有 seed
    for seed in seed_list:
        average_score_json_path = os.path.join(
            base_average_score_dir, 
            f"generate_images_seed_{seed}/{dataset_name}/{method}/ckpt-{ckpt}/average_scores.json"
        )
        
        if not os.path.exists(average_score_json_path):
            print(f"  Warning: File not found for seed {seed}: {average_score_json_path}")
            continue
        
        # 读取 average_score.json
        with open(average_score_json_path, 'r') as f:
            average_scores = json.load(f)
        
        print(f"  Seed {seed}:")
        # 收集每个指标的分数
        for metric in evaluation_metrics:
            if metric in average_scores:
                score = average_scores[metric]
                method_scores[metric].append(score)
                print(f"    {metric}: {score:.6f}")
            else:
                print(f"    {metric}: Not found")
    
    # 计算每个指标在所有 seed 上的平均值
    method_average_scores = {}
    print(f"\n  Average scores across all seeds:")
    for metric in evaluation_metrics:
        if metric in method_scores and len(method_scores[metric]) > 0:
            avg_score = np.mean(method_scores[metric])
            method_average_scores[metric] = avg_score
            print(f"    {metric}: {avg_score:.6f} (from {len(method_scores[metric])} seeds)")
        else:
            print(f"    {metric}: No data available")
    
    # 用 (method, ckpt) 作为 key，便于比较同一方法在不同 ckpt 下的性能
    all_method_scores[(method, ckpt)] = method_average_scores

# 打印汇总结果（含 ckpt 列）
print(f"\n{'='*80}")
print("SUMMARY: Average scores for each method (with ckpt) across all seeds")
print(f"{'='*80}")
print(f"{'Method':<60} {'ckpt':>6} " + " ".join([f"{m:>12}" for m in evaluation_metrics]))
print("-" * 80)

for (method, ckpt), scores in all_method_scores.items():
    method_display = method[:58] + ".." if len(method) > 60 else method
    score_str = " ".join([f"{scores.get(m, 0):>12.6f}" if m in scores else f"{'N/A':>12}" for m in evaluation_metrics])
    print(f"{method_display:<60} {ckpt:>6} {score_str}")

# # 保存结果到 JSON 文件（含 method 与 ckpt，便于查看不同 ckpt 性能）
# output_file = os.path.join(base_average_score_dir, "average_scores_summary.json")
# summary_for_json = [{"method": m, "ckpt": c, **scores} for (m, c), scores in all_method_scores.items()]
# with open(output_file, 'w') as f:
#     json.dump(summary_for_json, f, indent=4)
# print(f"\nResults saved to: {output_file}")


Processing method: sd-3-5-medium (ckpt: 0)
  Seed 42:
    pickscore: 0.861395
    imagereward: 0.780971
    unifiedreward: 3.309850
    hpsv3: 9.751047
    deqa: 4.118477
    aesthetic: 5.418618
    color-fidelity-metric: 12.463085
  Seed 123:
    pickscore: 0.861902
    imagereward: 0.738687
    unifiedreward: 3.307050
    hpsv3: 10.173014
    deqa: 4.118145
    aesthetic: 5.466564
    color-fidelity-metric: 12.500848
  Seed 456:
    pickscore: 0.863764
    imagereward: 0.794617
    unifiedreward: Not found
    hpsv3: 10.027917
    deqa: 4.038115
    aesthetic: 5.437598
    color-fidelity-metric: 12.561363
  Seed 789:
    pickscore: 0.859351
    imagereward: 0.771872
    unifiedreward: Not found
    hpsv3: 9.901634
    deqa: 4.065703
    aesthetic: 5.398283
    color-fidelity-metric: 12.570235
  Seed 1000:
    pickscore: 0.864250
    imagereward: 0.877444
    unifiedreward: Not found
    hpsv3: 10.273923
    deqa: 4.094316
    aesthetic: 5.468343
    color-fidelity-metric: 12.596865


In [ ]:
#### Mean Artifact Ratio ####
import os
import json
import numpy as np
from collections import defaultdict

base_model = "sd-3-5-medium"
rl_framework = "diffusion-dpo"
dataset_name = "partiprompts"
base_average_score_dir = f"/data_center/data2/dataset/chenwy/21164-data/{rl_framework}/{base_model}"
seed_list = [42, 123, 456, 789, 1000]
method_list = ["sd-3-5-medium", "FlowGRPO-PickScore", "pick-a-pic-v2-dpo_dataset_160000_pairs", "irl_top_512_images_no_anime_colorfulness_pickscore_0.02-hpdv3_all_lr_0.0002_ckpt_3200-dpo_top_512_images_no_anime_colorfulness_pickscore_0.02-hpdv3_all"]
ckpt_list = [0, 0, 4900, 450]

# 遍历每个方法
for method, ckpt in zip(method_list, ckpt_list):
    print(f"\n{'='*80}")
    print(f"Processing method: {method} (ckpt: {ckpt})")
    print(f"{'='*80}")
    
    # 存储当前方法所有 seed 的分数
    method_scores = defaultdict(list)
    
    # 遍历所有 seed
    for seed in seed_list:
        evaluation_results_path = os.path.join(
            base_average_score_dir, 
            f"generate_images_seed_{seed}/{dataset_name}/{method}/ckpt-{ckpt}/evaluation_results.jsonl"
        )
        
        if not os.path.exists(evaluation_results_path):
            print(f"  Warning: File not found for seed {seed}: {evaluation_results_path}")
            continue
        
        # 读取 evaluation_results.jsonl (JSONL 格式：每行一个 JSON 对象)
        evaluation_results = []
        with open(evaluation_results_path, 'r') as f:
            for line in f:
                line = line.strip()
                if line:  # 跳过空行
                    evaluation_results.append(json.loads(line))
        
        # 统计 diffdoctor 不为零的样本数
        total_samples = len(evaluation_results)
        non_zero_count = 0
        zero_count = 0
        missing_count = 0
        
        for sample in evaluation_results:
            if 'scores' in sample and 'diffdoctor' in sample['scores']:
                diffdoctor_value = sample['scores']['diffdoctor']
                if diffdoctor_value != 0:
                    non_zero_count += 1
                else:
                    zero_count += 1
            else:
                missing_count += 1
        
        print(f"  Seed {seed}:")
        print(f"    Total samples: {total_samples}")
        print(f"    diffdoctor != 0: {non_zero_count} ({non_zero_count/total_samples*100:.2f}%)")
        print(f"    diffdoctor == 0: {zero_count} ({zero_count/total_samples*100:.2f}%)")
        if missing_count > 0:
            print(f"    Missing diffdoctor: {missing_count} ({missing_count/total_samples*100:.2f}%)")
        
        # 存储统计结果
        method_scores['non_zero_count'].append(non_zero_count)
        method_scores['total_samples'].append(total_samples)
    
    # 计算所有 seed 的平均值
    if method_scores['total_samples']:
        avg_non_zero = np.mean(method_scores['non_zero_count'])
        avg_total = np.mean(method_scores['total_samples'])
        avg_percentage = (avg_non_zero / avg_total * 100) if avg_total > 0 else 0
        print(f"\n  Average across all seeds:")
        print(f"    Average non-zero samples: {avg_non_zero:.2f} out of {avg_total:.2f} ({avg_percentage:.2f}%)")